# IPL Cricket Match Simulation using Monte Carlo Methods

---

## What does this notebook do?

This notebook **simulates an IPL cricket match ball-by-ball** using real historical data from ESPNcricinfo.

Approach:
1. We look at how each batsman and bowler has performed historically (ball-by-ball)
2. We fit **probability distributions** to their performance
3. We then **simulate** thousands of matches by randomly sampling from these distributions
4. The result: win probabilities, expected scores, and realistic match outcomes

---

## What is a Monte Carlo Simulation?

A **Monte Carlo simulation** is a technique where we use **random sampling** to understand the behaviour of a system.

Think of it like this:
- You can't predict exactly what will happen on each ball of a cricket match
- But you *do* know the probabilities (e.g., Kohli hits a four ~13% of the time)
- So you simulate the match thousands of times using those probabilities
- Then you look at the overall pattern: "Team A wins 62% of the time, average score is 168"

---

## Our 5 Random Variables

| # | Variable | What it represents | Distribution |
|---|----------|-------------------|-------------|
| 1 | **X** — Balls before dismissal | How long a batsman survives | Geometric(p) |
| 2 | **R** — Runs per ball (given not out) | Scoring on each delivery | Categorical |
| 3 | **W** — Extras per ball | Wides/no-balls by bowler | Bernoulli(q) |
| 4 | **E** — Extra runs (given extra) | How many extra runs | Categorical |
| 5 | **S** — Strike rotation tendency | Tendency to take singles vs play dots | Bernoulli(s) |

All distributions are fitted **per player** and **per phase** (Powerplay / Middle / Death overs).

---

## Data

- **Training data:** IPL 2024 (all 74 matches, ball-by-ball) — used to fit distributions
- **Validation data:** IPL 2025 — used to test if our simulation produces realistic results
- **Source:** [ESPNcricinfo](https://www.espncricinfo.com/) via the [`cricdata`](https://github.com/arnavbonigala/cricdata) Python library

---
## Section 1: Install & Import Libraries

We need:
- **`cricdata`** — pulls ball-by-ball data from ESPNcricinfo
- **`pandas`** — data manipulation (think of it as Excel for Python)
- **`numpy`** — numerical operations and random sampling
- **`matplotlib` + `seaborn`** — plotting charts

In [ ]:
# Install the cricdata library (not pre-installed in Colab)
# -q means "quiet" — less output clutter
!pip install cricdata -q

In [ ]:
# Standard Python libraries for data science
import pandas as pd               # DataFrames — tabular data
import numpy as np                 # Arrays, random number generation
import matplotlib.pyplot as plt    # Plotting
import seaborn as sns              # Prettier plots on top of matplotlib
import time                        # To add delays between API calls
import os                          # To check if cached files exist
import warnings
warnings.filterwarnings('ignore')  # Suppress non-critical warnings

# The cricket data library
from cricdata import CricinfoClient

# Plot style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

print("All libraries loaded successfully!")

---
## Section 2: Data Collection

We pull **ball-by-ball data** from ESPNcricinfo for multiple IPL seasons:

| Season | Purpose | Weight | Why |
|--------|---------|--------|-----|
| IPL 2021 | Training | 0.3 | Oldest — players may have changed style |
| IPL 2022 | Training | 0.5 | Older data, still useful |
| IPL 2023 | Training | 0.7 | Recent and relevant |
| IPL 2024 | Training | 1.0 | Most recent — highest weight |
| IPL 2025 | Validation | — | Held out to test the model |

### Why multiple seasons?
With only 1 season (~74 matches), many players have too few balls to build reliable distributions.
With 4 seasons (~280+ matches), every regular player has hundreds of deliveries — much better estimates.

### Recency weighting
A player's 2024 form matters more than 2021. Each ball is weighted by its season:
- A 2024 ball counts as **1.0** data point
- A 2021 ball counts as **0.3** data point

This naturally phases out old data without discarding it entirely.

### Caching
Data is saved to CSV after the first pull, so re-runs are instant.

In [ ]:
# Initialize the ESPNcricinfo client
ci = CricinfoClient()

# ----- IPL Series Slugs -----
# Training data: multiple seasons for richer player statistics
# Validation data: IPL 2025 (held out)

TRAINING_SERIES = {
    "IPL 2021": {"slug": "ipl-2021-1210595", "weight": 0.3},
    "IPL 2022": {"slug": "ipl-2022-1298423", "weight": 0.5},
    "IPL 2023": {"slug": "ipl-2023-1345038", "weight": 0.7},
    "IPL 2024": {"slug": "ipl-2024-1410320", "weight": 1.0},
}

VALIDATION_SERIES = {
    "IPL 2025": {"slug": "ipl-2025-1449924", "weight": 1.0},
}

print("Client ready. Training series:")
for name, info in TRAINING_SERIES.items():
    print(f"  {name}: {info['slug']} (weight: {info['weight']})")
print(f"\nValidation: IPL 2025")

In [ ]:
def parse_ball(ball, match_title, match_slug, season):
    """
    Flatten one delivery (ball) from the API's nested JSON into a simple dictionary.

    The API returns deeply nested data like:
        ball -> batsman -> athlete -> displayName
    We extract just the fields we need into a flat row.

    Parameters:
        ball (dict): One delivery from the API
        match_title (str): e.g. "1st Match"
        match_slug (str): URL identifier for the match
        season (str): e.g. "IPL 2024"

    Returns:
        dict: A flat dictionary with one key per column
    """
    over = ball.get("over", {})
    batsman = ball.get("batsman", {}).get("athlete", {})
    bowler = ball.get("bowler", {}).get("athlete", {})
    dismissal = ball.get("dismissal", {})
    play_type = ball.get("playType", {})
    innings_info = ball.get("innings", {})
    batting_team = ball.get("team", {})
    fielder = dismissal.get("fielder", {}).get("athlete", {})

    return {
        # Match info
        "season": season,
        "match": match_title,
        "match_slug": match_slug,
        "date": ball.get("date", ""),

        # Innings
        "innings": ball.get("period", ""),
        "innings_text": ball.get("periodText", ""),
        "batting_team": batting_team.get("name", ""),
        "batting_team_abbr": batting_team.get("abbreviation", ""),

        # Over & ball position
        "over_number": over.get("number", ""),
        "ball_in_over": over.get("ball", ""),
        "overs": over.get("overs", ""),

        # Players
        "batsman": batsman.get("displayName", ""),
        "batsman_id": batsman.get("id", ""),
        "bowler": bowler.get("displayName", ""),
        "bowler_id": bowler.get("id", ""),

        # Runs & play type
        "runs_scored": ball.get("scoreValue", 0),
        "play_type": play_type.get("description", ""),

        # Extras tracking
        "wide_in_over": over.get("wide", 0),
        "noball_in_over": over.get("noBall", 0),
        "byes_in_over": over.get("byes", 0),
        "legbyes_in_over": over.get("legByes", 0),

        # Dismissal info
        "is_wicket": dismissal.get("dismissal", False),
        "dismissal_type": dismissal.get("type", ""),
        "dismissed_batsman": dismissal.get("batsman", {}).get("athlete", {}).get("displayName", ""),
        "fielder": fielder.get("displayName", ""),

        # Cumulative innings state (running totals)
        "total_runs": innings_info.get("runs", ""),
        "total_wickets": innings_info.get("wickets", ""),
        "total_balls": innings_info.get("balls", ""),
        "run_rate": innings_info.get("runRate", ""),
        "remaining_balls": innings_info.get("remainingBalls", ""),
        "target": innings_info.get("target", 0),

        # Short text description
        "short_text": ball.get("shortText", ""),
    }


print("parse_ball() function defined.")

In [ ]:
def fetch_season_data(ci, series_slug, season_name, max_matches=None):
    """
    Fetch ball-by-ball data for an entire IPL season.

    Steps:
      1. Get the list of all matches (fixtures) in the season
      2. For each match, fetch ball-by-ball data
      3. Parse each ball into a flat row
      4. Return all rows as a list of dicts

    Parameters:
        ci: CricinfoClient instance
        series_slug (str): e.g. "ipl-2024-1410320"
        season_name (str): e.g. "IPL 2024" (for labelling)
        max_matches (int or None): Limit matches for testing. None = all.

    Returns:
        list[dict]: All deliveries as flat row dicts
    """
    # Step 1: Get fixtures
    fixtures = ci.series_fixtures(series_slug)
    matches = fixtures["content"]["matches"]
    if max_matches:
        matches = matches[:max_matches]

    print(f"\n{season_name}: {len(matches)} matches to fetch")

    all_rows = []
    failed = []

    for i, m in enumerate(matches):
        series_s = f"{m['series']['slug']}-{m['series']['objectId']}"
        match_s = f"{m['slug']}-{m['objectId']}"
        match_title = m.get("title", m["slug"])

        try:
            balls = ci.match_ball_by_ball(series_s, match_s)
            count = 0
            for innings in balls:
                for ball in innings:
                    all_rows.append(parse_ball(ball, match_title, match_s, season_name))
                    count += 1
            print(f"  [{i+1}/{len(matches)}] {match_title}: {count} balls")
        except Exception as e:
            failed.append(match_title)
            print(f"  [{i+1}/{len(matches)}] {match_title}: FAILED ({e})")

        time.sleep(1)  # 1s delay to avoid rate-limiting

    print(f"\n{season_name} done: {len(all_rows)} deliveries from {len(matches) - len(failed)} matches")
    if failed:
        print(f"  Failed: {failed}")

    return all_rows


print("fetch_season_data() function defined.")

In [ ]:
# ============================================================
#  FETCH DATA (or load from cache)
# ============================================================
# First run: ~8-10 minutes (fetching ~280 matches across 4 seasons)
# Subsequent runs: instant (loading from CSV)
# ============================================================

TRAIN_CSV = "ipl_train_multiyear.csv"
VAL_CSV = "ipl_2025_ball_by_ball.csv"

# --- Training data: IPL 2021-2024 ---
if os.path.exists(TRAIN_CSV):
    print(f"Loading cached training data from {TRAIN_CSV}...")
    df_train = pd.read_csv(TRAIN_CSV)
    print(f"  Loaded {len(df_train)} deliveries from {df_train['season'].nunique()} seasons")
else:
    print("Fetching training data from ESPNcricinfo...")
    print("This fetches 4 IPL seasons (~280 matches). Takes ~8-10 min.\n")
    all_training_rows = []

    for season_name, info in TRAINING_SERIES.items():
        rows = fetch_season_data(ci, info["slug"], season_name)
        # Add recency weight to each row
        for row in rows:
            row["weight"] = info["weight"]
        all_training_rows.extend(rows)

    df_train = pd.DataFrame(all_training_rows)
    df_train.to_csv(TRAIN_CSV, index=False)
    print(f"\nSaved {len(df_train)} total training deliveries to {TRAIN_CSV}")

# --- Validation data: IPL 2025 ---
if os.path.exists(VAL_CSV):
    print(f"\nLoading cached validation data from {VAL_CSV}...")
    df_val = pd.read_csv(VAL_CSV)
    print(f"  Loaded {len(df_val)} deliveries")
else:
    print("\nFetching IPL 2025 data from ESPNcricinfo...")
    rows = fetch_season_data(ci, VALIDATION_SERIES["IPL 2025"]["slug"], "IPL 2025")
    for row in rows:
        row["weight"] = 1.0
    df_val = pd.DataFrame(rows)
    df_val.to_csv(VAL_CSV, index=False)
    print(f"  Saved to {VAL_CSV}")

# Ensure weight column exists (for cached files from older runs)
if "weight" not in df_train.columns:
    # Assign weights based on season
    weight_map = {name: info["weight"] for name, info in TRAINING_SERIES.items()}
    df_train["weight"] = df_train["season"].map(weight_map).fillna(0.5)

if "weight" not in df_val.columns:
    df_val["weight"] = 1.0

print(f"\nTraining data: {df_train.shape[0]:,} deliveries, {df_train['match'].nunique()} matches")
print(f"  Seasons: {dict(df_train['season'].value_counts())}")
print(f"Validation data: {df_val.shape[0]:,} deliveries, {df_val['match'].nunique()} matches")

---
## Section 3: Data Cleaning & Feature Engineering

The raw data needs some preparation before we can use it for modelling:

1. **Add `phase` column** — In T20 cricket, the game has three distinct phases:
   - **Powerplay (PP):** Overs 1-6 — fielding restrictions, batsmen attack
   - **Middle:** Overs 7-15 — balanced play, building innings
   - **Death:** Overs 16-20 — all-out attack, big hits, high risk

2. **Add `is_extra` column** — Flag balls that are wides or no-balls

3. **Add `is_boundary` column** — Flag 4s and 6s

4. **Filter out super overs and abandoned matches**

In [ ]:
def clean_and_engineer(df):
    """
    Clean the raw ball-by-ball DataFrame and add derived features.

    New columns added:
        - phase: 'powerplay', 'middle', or 'death'
        - is_extra: True if the delivery was a wide or no-ball
        - is_boundary: True if 4 or 6 was scored
        - is_dot: True if 0 runs scored (and not a wicket-extra)
    """
    df = df.copy()

    # Ensure numeric types
    df["over_number"] = pd.to_numeric(df["over_number"], errors="coerce")
    df["runs_scored"] = pd.to_numeric(df["runs_scored"], errors="coerce").fillna(0).astype(int)

    # Filter: only innings 1 and 2 (remove super overs = innings 3)
    df = df[df["innings"].isin([1, 2])].copy()

    # Phase assignment based on over number
    # Over 1-6 = Powerplay, 7-15 = Middle, 16-20 = Death
    def assign_phase(over_num):
        if over_num <= 6:
            return "powerplay"
        elif over_num <= 15:
            return "middle"
        else:
            return "death"

    df["phase"] = df["over_number"].apply(assign_phase)

    # Extra detection: a ball is an extra if play_type mentions wide or no ball
    df["is_extra"] = df["play_type"].str.contains("wide|no ball", case=False, na=False)

    # Boundary detection
    df["is_boundary"] = df["runs_scored"].isin([4, 6])

    # Dot ball detection (0 runs, not an extra)
    df["is_dot"] = (df["runs_scored"] == 0) & (~df["is_extra"])

    # Drop rows with missing critical data
    df = df.dropna(subset=["batsman", "bowler", "over_number"]).copy()

    return df


# Apply cleaning to both datasets
df_train = clean_and_engineer(df_train)
df_val = clean_and_engineer(df_val)

print(f"Training data after cleaning: {len(df_train)} deliveries")
print(f"Validation data after cleaning: {len(df_val)} deliveries")

In [ ]:
# Let's look at the data!
print("=== Sample rows from training data ===")
print(f"Columns ({len(df_train.columns)}): {list(df_train.columns)}")
print()

# Show a few key columns
display_cols = [
    "season", "match", "innings", "phase", "overs",
    "batsman", "bowler", "runs_scored", "play_type",
    "is_wicket", "is_extra", "short_text"
]
df_train[display_cols].head(15)

In [ ]:
# Quick summary statistics
print("=== Training Data Summary ===")
print(f"Total deliveries : {len(df_train):,}")
print(f"Total matches    : {df_train['match'].nunique()}")
print(f"Seasons          : {sorted(df_train['season'].unique())}")
print(f"Unique batsmen   : {df_train['batsman'].nunique()}")
print(f"Unique bowlers   : {df_train['bowler'].nunique()}")
print(f"Total wickets    : {df_train['is_wicket'].sum()}")
print(f"Total extras     : {df_train['is_extra'].sum()}")
print(f"Total boundaries : {df_train['is_boundary'].sum()}")
print(f"\nDeliveries per season:")
print(df_train.groupby("season")["weight"].agg(["count", "first"]).rename(columns={"count": "deliveries", "first": "weight"}))
print(f"\nPhase distribution:")
print(df_train["phase"].value_counts())

---
## Section 4: Exploratory Data Analysis (EDA)

Before building our model, let's **visualize the data** to understand scoring patterns in IPL cricket.

This helps us:
- Verify our data looks correct
- See which distributions might be appropriate
- Understand phase-wise differences

In [ ]:
# ---- Plot 1: Runs per delivery distribution ----
# This shows how often each outcome (0, 1, 2, 3, 4, 6 runs) occurs

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
run_counts = df_train["runs_scored"].value_counts().sort_index()
run_pcts = (run_counts / len(df_train) * 100).round(1)

axes[0].bar(run_pcts.index.astype(str), run_pcts.values, color="steelblue", edgecolor="black")
for i, (idx, val) in enumerate(run_pcts.items()):
    axes[0].text(i, val + 0.5, f"{val}%", ha="center", fontsize=10)
axes[0].set_xlabel("Runs Scored")
axes[0].set_ylabel("Percentage of Deliveries")
axes[0].set_title("Runs per Delivery — Overall")

# Phase-wise distribution
phase_run = df_train.groupby(["phase", "runs_scored"]).size().unstack(fill_value=0)
phase_run_pct = phase_run.div(phase_run.sum(axis=1), axis=0) * 100
phase_run_pct.loc[["powerplay", "middle", "death"]].T.plot(
    kind="bar", ax=axes[1], edgecolor="black"
)
axes[1].set_xlabel("Runs Scored")
axes[1].set_ylabel("Percentage")
axes[1].set_title("Runs per Delivery — By Phase")
axes[1].legend(title="Phase")
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print("Key insight: Death overs have more 4s and 6s, Powerplay has fewer dots.")

In [ ]:
# ---- Plot 2: Wickets per innings & Extras per innings ----

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Wickets per innings
wickets_per_inn = df_train.groupby(["match", "innings"])["is_wicket"].sum()
axes[0].hist(wickets_per_inn, bins=range(0, 12), edgecolor="black", color="salmon", align="left")
axes[0].set_xlabel("Wickets Lost")
axes[0].set_ylabel("Number of Innings")
axes[0].set_title(f"Wickets per Innings (mean: {wickets_per_inn.mean():.1f})")
axes[0].axvline(wickets_per_inn.mean(), color="red", linestyle="--", label=f"Mean: {wickets_per_inn.mean():.1f}")
axes[0].legend()

# Extras per innings
extras_per_inn = df_train.groupby(["match", "innings"])["is_extra"].sum()
axes[1].hist(extras_per_inn, bins=range(0, 20), edgecolor="black", color="lightgreen", align="left")
axes[1].set_xlabel("Extras")
axes[1].set_ylabel("Number of Innings")
axes[1].set_title(f"Extras per Innings (mean: {extras_per_inn.mean():.1f})")
axes[1].axvline(extras_per_inn.mean(), color="green", linestyle="--", label=f"Mean: {extras_per_inn.mean():.1f}")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 3: Run rate progression by over ----
# This is the classic "batting phases" chart

rpo = df_train.groupby("over_number")["runs_scored"].mean() * 6  # runs per over

fig, ax = plt.subplots(figsize=(12, 5))
colors = ["#2196F3" if o <= 6 else "#FF9800" if o <= 15 else "#F44336" for o in rpo.index]
ax.bar(rpo.index, rpo.values, color=colors, edgecolor="black")

# Phase labels
ax.axvspan(0.5, 6.5, alpha=0.1, color="blue", label="Powerplay")
ax.axvspan(6.5, 15.5, alpha=0.1, color="orange", label="Middle")
ax.axvspan(15.5, 20.5, alpha=0.1, color="red", label="Death")

ax.set_xlabel("Over Number")
ax.set_ylabel("Average Runs per Over")
ax.set_title("Run Rate Progression by Over — IPL 2024")
ax.set_xticks(range(1, 21))
ax.legend()

plt.tight_layout()
plt.show()

print("Classic T20 pattern: aggressive start, consolidation, then explosion in death overs.")

In [ ]:
# ---- Top Batsmen and Bowlers ----

print("=== Top 10 Batsmen by Runs (IPL 2024 Training Data) ===")
top_bat = df_train.groupby("batsman").agg(
    balls=('runs_scored', 'count'),
    runs=('runs_scored', 'sum'),
    dismissals=('is_wicket', 'sum'),
    fours=('runs_scored', lambda x: (x == 4).sum()),
    sixes=('runs_scored', lambda x: (x == 6).sum()),
).sort_values("runs", ascending=False)
top_bat["SR"] = (top_bat["runs"] / top_bat["balls"] * 100).round(1)
print(top_bat.head(10).to_string())

print("\n=== Top 10 Bowlers by Wickets ===")
top_bowl = df_train.groupby("bowler").agg(
    balls=('runs_scored', 'count'),
    runs_conceded=('runs_scored', 'sum'),
    wickets=('is_wicket', 'sum'),
    extras=('is_extra', 'sum'),
).sort_values("wickets", ascending=False)
top_bowl["econ"] = (top_bowl["runs_conceded"] / top_bowl["balls"] * 6).round(2)
print(top_bowl.head(10).to_string())

---
## Section 5: Fitting the 5 Probability Distributions

Now we compute the parameters for each of our 5 random variables from the training data.

### Quick refresher on the distributions:

| Distribution | What it is | Example |
|---|---|---|
| **Geometric(p)** | Number of independent trials until first "success" (here, success = getting out). Parameter **p** = probability of getting out on any given ball. | p=0.04 means batsman gets out once every 25 balls on average |
| **Categorical** | Like rolling a weighted die. Each outcome has its own probability. Probabilities must sum to 1. | P(0)=0.35, P(1)=0.35, P(2)=0.08, P(4)=0.14, P(6)=0.08 |
| **Bernoulli(q)** | A single coin flip. Outcome is either 0 or 1. Parameter **q** = probability of getting 1. | q=0.05 means 5% chance of a wide/no-ball |

All parameters are computed **per player** and **per phase** where possible.

In [ ]:
# ============================================================
#  VARIABLE 1: X ~ Geometric(p) — Batsman Survival
# ============================================================
# p = probability of getting out on any given ball
# Computed per batsman, per phase, using RECENCY-WEIGHTED counts
#
# Instead of counting each ball equally, recent seasons count more:
#   2024 ball = weight 1.0, 2021 ball = weight 0.3
# ============================================================

legit_balls = df_train[~df_train["is_extra"]].copy()

# Weighted aggregation: sum of weights for balls faced, sum of weights for dismissals
survival_params = legit_balls.groupby(["batsman", "phase"]).agg(
    balls_faced_w=('weight', 'sum'),                                    # weighted ball count
    dismissals_w=('weight', lambda x: (x * legit_balls.loc[x.index, "is_wicket"]).sum()),  # weighted dismissals
    balls_faced=('weight', 'count'),  # raw count for display
).reset_index()

# Fix: compute dismissals_w properly using a merge approach
_wk = legit_balls[["batsman", "phase", "weight", "is_wicket"]].copy()
_wk["wk_weight"] = _wk["weight"] * _wk["is_wicket"].astype(float)

survival_params = _wk.groupby(["batsman", "phase"]).agg(
    balls_faced_w=('weight', 'sum'),
    dismissals_w=('wk_weight', 'sum'),
    balls_faced=('weight', 'count'),
    dismissals=('is_wicket', 'sum'),
).reset_index()

# Bayesian smoothing with stronger prior
PRIOR_DISMISSALS = 2
PRIOR_BALLS = 30
survival_params["p"] = (
    (survival_params["dismissals_w"] + PRIOR_DISMISSALS) / (survival_params["balls_faced_w"] + PRIOR_BALLS)
)

# Global (non-phase) fallback
survival_global = _wk.groupby("batsman").agg(
    balls_faced_w=('weight', 'sum'),
    dismissals_w=('wk_weight', 'sum'),
).reset_index()
survival_global["p"] = (survival_global["dismissals_w"] + PRIOR_DISMISSALS) / (survival_global["balls_faced_w"] + PRIOR_BALLS)

print("Variable 1: Geometric survival parameters computed (recency-weighted).")
print(f"  Entries: {len(survival_params)} (batsman x phase combinations)")
print(f"  Smoothing prior: {PRIOR_DISMISSALS} dismissals / {PRIOR_BALLS} balls")
print()

# Show for a sample batsman
sample_bat = survival_params[survival_params["balls_faced"] > 100].iloc[0]["batsman"]
print(f"Example — {sample_bat}:")
display = survival_params[survival_params["batsman"] == sample_bat][["phase", "balls_faced", "dismissals", "p"]].copy()
display["expected_balls_before_out"] = (1 / display["p"]).round(1)
print(display.to_string(index=False))

In [ ]:
# ============================================================
#  VARIABLE 2: R ~ Categorical — Runs per ball (given NOT out)
# ============================================================
# Uses LEAGUE-PROPORTIONAL smoothing instead of flat Laplace.
#
# Flat Laplace: adds +1 to every category → inflates rare events (4s, 6s)
# League-proportional: adds counts proportional to actual league distribution
#   → pulls small-sample players toward league average, not toward uniform
# ============================================================

survived = legit_balls[legit_balls["is_wicket"] == False].copy()
RUN_VALUES = [0, 1, 2, 3, 4, 6]

# Step 1: Compute league-wide distribution (weighted)
_league_counts = {}
for r in RUN_VALUES:
    _league_counts[r] = survived.loc[survived["runs_scored"] == r, "weight"].sum()
_league_total = sum(_league_counts.values())
LEAGUE_RUN_DIST = {r: _league_counts[r] / _league_total for r in RUN_VALUES}

# Smoothing strength: equivalent to adding ~15 "pseudo-balls" from the league distribution
# This means a player needs ~30+ real balls before their own data dominates
SMOOTHING_STRENGTH = 15

LEAGUE_PSEUDO_COUNTS = {r: LEAGUE_RUN_DIST[r] * SMOOTHING_STRENGTH for r in RUN_VALUES}

print("League-wide run distribution (weighted):")
for r in RUN_VALUES:
    print(f"  {r} runs: {LEAGUE_RUN_DIST[r]*100:.1f}% (pseudo-count: {LEAGUE_PSEUDO_COUNTS[r]:.1f})")
print()


def compute_scoring_probs_weighted(group):
    """
    Compute scoring probabilities using recency-weighted counts
    and league-proportional smoothing.
    """
    weighted_counts = {}
    for r in RUN_VALUES:
        mask = group["runs_scored"] == r
        weighted_counts[r] = group.loc[mask, "weight"].sum() + LEAGUE_PSEUDO_COUNTS[r]
    total = sum(weighted_counts.values())
    return {f"p_{r}": weighted_counts[r] / total for r in RUN_VALUES}


scoring_params = (
    survived.groupby(["batsman", "phase"])
    .apply(compute_scoring_probs_weighted, include_groups=False)
    .apply(pd.Series)
    .reset_index()
)

scoring_global = (
    survived.groupby("batsman")
    .apply(compute_scoring_probs_weighted, include_groups=False)
    .apply(pd.Series)
    .reset_index()
)

print("Variable 2: Categorical scoring parameters computed (league-proportional smoothing).")
print(f"  Entries: {len(scoring_params)} (batsman x phase combinations)")
print()
print(f"Example — {sample_bat}:")
print(scoring_params[scoring_params["batsman"] == sample_bat].to_string(index=False))

In [ ]:
# ============================================================
#  VARIABLE 3: W ~ Bernoulli(q) — Extras per ball (per bowler)
# ============================================================
# q = probability that a given delivery is a wide or no-ball
# Computed per bowler using recency-weighted counts.
# ============================================================

_extras_wt = df_train[["bowler", "is_extra", "weight"]].copy()
_extras_wt["extra_weight"] = _extras_wt["weight"] * _extras_wt["is_extra"].astype(float)

extras_params = _extras_wt.groupby("bowler").agg(
    total_balls_w=('weight', 'sum'),
    extras_w=('extra_weight', 'sum'),
).reset_index()

extras_params["q"] = (extras_params["extras_w"] + 1) / (extras_params["total_balls_w"] + 20)

LEAGUE_EXTRA_RATE = _extras_wt["extra_weight"].sum() / _extras_wt["weight"].sum()

print("Variable 3: Bernoulli extras parameters computed (recency-weighted).")
print(f"  League average extras rate: {LEAGUE_EXTRA_RATE:.3f} ({LEAGUE_EXTRA_RATE*100:.1f}%)")

In [ ]:
# ============================================================
#  VARIABLE 4: E ~ Categorical — Extra runs (given extra occurs)
# ============================================================
# When a wide or no-ball happens, how many runs come off it?
# Usually 1 (just the extra), but sometimes 2+ (batsman scores off a no-ball)
# ============================================================

extra_balls = df_train[df_train["is_extra"]].copy()

extra_runs_dist = extra_balls["runs_scored"].value_counts(normalize=True).sort_index()
EXTRA_RUNS_VALUES = extra_runs_dist.index.tolist()
EXTRA_RUNS_PROBS = extra_runs_dist.values.tolist()

print("Variable 4: Extra runs distribution:")
for val, prob in zip(EXTRA_RUNS_VALUES, EXTRA_RUNS_PROBS):
    print(f"  {val} run(s): {prob:.3f} ({prob*100:.1f}%)")

In [ ]:
# ============================================================
#  VARIABLE 5: S ~ Bernoulli(s) — Strike rotation tendency
# ============================================================
# Recency-weighted version.
# ============================================================

rotation_balls = survived[survived["runs_scored"].isin([0, 1])].copy()
rotation_balls["single_weight"] = rotation_balls["weight"] * rotation_balls["runs_scored"].astype(float)

rotation_params = rotation_balls.groupby(["batsman", "phase"]).agg(
    dot_or_single_w=('weight', 'sum'),
    singles_w=('single_weight', 'sum'),
    dot_or_single=('weight', 'count'),
    singles=('runs_scored', 'sum'),
).reset_index()

rotation_params["s"] = (rotation_params["singles_w"] + 1) / (rotation_params["dot_or_single_w"] + 2)

rotation_global = rotation_balls.groupby("batsman").agg(
    dot_or_single_w=('weight', 'sum'),
    singles_w=('single_weight', 'sum'),
).reset_index()
rotation_global["s"] = (rotation_global["singles_w"] + 1) / (rotation_global["dot_or_single_w"] + 2)

print("Variable 5: Strike rotation parameters computed (recency-weighted).")
print(f"  Entries: {len(rotation_params)} (batsman x phase combinations)")
print()
print(f"Example — {sample_bat}:")
print(rotation_params[rotation_params["batsman"] == sample_bat][["phase", "dot_or_single", "singles", "s"]].to_string(index=False))

In [ ]:
# ============================================================
#  PARAMETER LOOKUP — FAST CACHED VERSION
# ============================================================

# League-wide fallback values
LEAGUE_P = (legit_balls["is_wicket"].astype(float) * legit_balls["weight"]).sum()
LEAGUE_P = (LEAGUE_P + PRIOR_DISMISSALS) / (legit_balls["weight"].sum() + PRIOR_BALLS)

LEAGUE_SCORING = [LEAGUE_RUN_DIST[r] for r in RUN_VALUES]

_all_rotation = rotation_balls
LEAGUE_S = (_all_rotation["single_weight"].sum() + 1) / (_all_rotation["weight"].sum() + 2)

# ---- Build fast lookup dicts ----
_surv_cache = {}
for _, row in survival_params.iterrows():
    _surv_cache[(row["batsman"], row["phase"])] = row["p"]
_surv_global = dict(zip(survival_global["batsman"], survival_global["p"]))

_score_cache = {}
for _, row in scoring_params.iterrows():
    _score_cache[(row["batsman"], row["phase"])] = [row[f"p_{r}"] for r in RUN_VALUES]
_score_global = {}
for _, row in scoring_global.iterrows():
    _score_global[row["batsman"]] = [row[f"p_{r}"] for r in RUN_VALUES]

_rot_cache = {}
for _, row in rotation_params.iterrows():
    _rot_cache[(row["batsman"], row["phase"])] = row["s"]
_rot_global = dict(zip(rotation_global["batsman"], rotation_global["s"]))

_extras_cache = dict(zip(extras_params["bowler"], extras_params["q"]))


def get_survival_p(batsman, phase):
    return _surv_cache.get((batsman, phase), _surv_global.get(batsman, LEAGUE_P))

def get_scoring_probs(batsman, phase):
    return _score_cache.get((batsman, phase), _score_global.get(batsman, LEAGUE_SCORING))

def get_rotation_s(batsman, phase):
    return _rot_cache.get((batsman, phase), _rot_global.get(batsman, LEAGUE_S))

def get_extras_q(bowler):
    return _extras_cache.get(bowler, LEAGUE_EXTRA_RATE)


print("Fast cached parameter lookups built.")
print(f"  Survival cache:  {len(_surv_cache)} entries")
print(f"  Scoring cache:   {len(_score_cache)} entries")
print(f"  Rotation cache:  {len(_rot_cache)} entries")
print(f"  Extras cache:    {len(_extras_cache)} entries")
print()
print(f"League-wide fallback values:")
print(f"  Dismissal rate (p): {LEAGUE_P:.4f} (1 out every {1/LEAGUE_P:.0f} balls)")
print(f"  Extras rate (q):    {LEAGUE_EXTRA_RATE:.4f}")
print(f"  Strike rotation (s):{LEAGUE_S:.4f}")

---
## Section 6: Team Selection

Pick two IPL teams for the simulation. You can:
- **Choose your own teams and players** from the dropdown
- **Press Enter** at any prompt to accept the default

### How defaults work:
- Default teams: **Chennai Super Kings vs Mumbai Indians**
- Default playing XI: Top 11 players by balls faced for that team in IPL 2024
- Default batting order: Sorted by typical batting position from data
- Default bowlers: Top 5 by overs bowled for that team

In [ ]:
# Show available teams
teams = sorted(df_train["batting_team"].unique())
print("Available IPL teams in the dataset:")
for i, t in enumerate(teams, 1):
    print(f"  {i}. {t}")

In [ ]:
# ============================================================
#  TEAM & PLAYER SELECTION
# ============================================================

DEFAULT_TEAM_A = "Chennai Super Kings"
DEFAULT_TEAM_B = "Mumbai Indians"


def get_team_players(team_name, df, role="bat"):
    """
    Get the best players for a team from the data.

    For batting: sorted by total balls faced (most experienced first)
    For bowling: sorted by total balls bowled
    """
    if role == "bat":
        team_df = df[df["batting_team"] == team_name]
        players = team_df.groupby("batsman").agg(
            balls=('runs_scored', 'count'),
            runs=('runs_scored', 'sum'),
        ).sort_values("balls", ascending=False)
        players["SR"] = (players["runs"] / players["balls"] * 100).round(1)
        return players
    else:
        # Bowlers: people who bowled AGAINST this team don't count.
        # We need people who bowled FOR this team = bowled when the OTHER team was batting.
        bowling_df = df[df["batting_team"] != team_name]
        # But we need to know which bowlers belong to this team.
        # Heuristic: players who batted for this team are on this team.
        team_batsmen = set(df[df["batting_team"] == team_name]["batsman"].unique())
        bowling_df = bowling_df[bowling_df["bowler"].isin(
            # Get bowlers from matches where this team played
            bowling_df["bowler"].unique()
        )]
        # Actually, simpler: look at matches this team played, get bowlers when opponent was batting
        team_matches = df[df["batting_team"] == team_name]["match"].unique()
        opp_batting = df[(df["match"].isin(team_matches)) & (df["batting_team"] != team_name)]
        players = opp_batting.groupby("bowler").agg(
            balls=('runs_scored', 'count'),
            runs_conceded=('runs_scored', 'sum'),
            wickets=('is_wicket', 'sum'),
        ).sort_values("balls", ascending=False)
        players["econ"] = (players["runs_conceded"] / players["balls"] * 6).round(2)
        return players


def select_team(team_label, default_team, df):
    """
    Interactive team selection flow.
    Returns: (team_name, batting_order_list, bowler_list)
    """
    # Step 1: Pick team
    print(f"\n{'='*50}")
    print(f"  {team_label} SELECTION")
    print(f"{'='*50}")
    team_input = input(f"Enter team name for {team_label} (default: {default_team}): ").strip()
    team_name = team_input if team_input else default_team

    # Fuzzy match: find closest team name
    matched = [t for t in teams if team_name.lower() in t.lower()]
    if matched:
        team_name = matched[0]
    elif team_name not in teams:
        print(f"  Team '{team_name}' not found. Using {default_team}.")
        team_name = default_team
    print(f"  Selected: {team_name}")

    # Step 2: Show available players
    batters = get_team_players(team_name, df, "bat")
    bowlers = get_team_players(team_name, df, "bowl")

    print(f"\n  Top batsmen for {team_name}:")
    for i, (name, row) in enumerate(batters.head(15).iterrows(), 1):
        print(f"    {i:2d}. {name:<25s}  {row['balls']:3.0f} balls, {row['runs']:3.0f} runs, SR {row['SR']}")

    print(f"\n  Top bowlers for {team_name}:")
    for i, (name, row) in enumerate(bowlers.head(8).iterrows(), 1):
        print(f"    {i:2d}. {name:<25s}  {row['balls']:3.0f} balls, {row['wickets']:.0f} wkts, econ {row['econ']}")

    # Step 3: Pick batting order (or default)
    print(f"\n  Enter 11 batsman names (comma-separated) for batting order.")
    print(f"  Press Enter for default (top 11 by balls faced).")
    bat_input = input("  Batting order: ").strip()

    if bat_input:
        batting_order = [b.strip() for b in bat_input.split(",")][:11]
        # Validate names
        valid_names = set(batters.index) | set(bowlers.index)
        batting_order = [b for b in batting_order if any(b.lower() in v.lower() for v in valid_names)]
        # Fuzzy match
        resolved = []
        for b in batting_order:
            match = [v for v in valid_names if b.lower() in v.lower()]
            resolved.append(match[0] if match else b)
        batting_order = resolved
    else:
        batting_order = batters.head(11).index.tolist()

    # Pad to 11 if needed
    if len(batting_order) < 11:
        remaining = [b for b in batters.index if b not in batting_order]
        batting_order.extend(remaining[:11 - len(batting_order)])
    batting_order = batting_order[:11]

    # Step 4: Pick bowlers (or default)
    print(f"\n  Enter 5 bowler names (comma-separated).")
    print(f"  Press Enter for default (top 5 by balls bowled).")
    bowl_input = input("  Bowlers: ").strip()

    if bowl_input:
        bowler_list = [b.strip() for b in bowl_input.split(",")][:5]
        resolved = []
        for b in bowler_list:
            match = [v for v in bowlers.index if b.lower() in v.lower()]
            resolved.append(match[0] if match else b)
        bowler_list = resolved
    else:
        bowler_list = bowlers.head(5).index.tolist()

    if len(bowler_list) < 5:
        remaining = [b for b in bowlers.index if b not in bowler_list]
        bowler_list.extend(remaining[:5 - len(bowler_list)])
    bowler_list = bowler_list[:5]

    # Summary
    print(f"\n  {team_name} — Final Selection:")
    print(f"  Batting order:")
    for i, b in enumerate(batting_order, 1):
        tag = " (B)" if b in bowler_list else ""
        print(f"    {i:2d}. {b}{tag}")
    print(f"  Bowlers: {', '.join(bowler_list)}")

    return team_name, batting_order, bowler_list


print("Team selection function defined.")

In [ ]:
# ---- Run team selection (interactive) ----

team_a_name, team_a_batting, team_a_bowlers = select_team("TEAM A", DEFAULT_TEAM_A, df_train)
team_b_name, team_b_batting, team_b_bowlers = select_team("TEAM B", DEFAULT_TEAM_B, df_train)

print(f"\n{'='*50}")
print(f"  MATCH: {team_a_name} vs {team_b_name}")
print(f"{'='*50}")

---
## Section 7: Simulation Engine

This is the core of the notebook. We simulate a cricket match **ball by ball**.

### How one ball works:

```
For each delivery:
  1. Determine the phase (powerplay / middle / death) from the over number
  2. Check for EXTRAS: flip Bernoulli(q) for the bowler
     → If extra: add runs, re-bowl the ball (doesn't count as a legal delivery)
  3. Check for WICKET: flip Bernoulli(p) for the batsman in this phase
     → If out: new batsman comes in
  4. If not out, SAMPLE RUNS:
     → First decide if it's a dot/single (using strike rotation Bernoulli(s))
     → Or a boundary (from Categorical distribution)
  5. Update score, rotate strike if needed
```

### Terminology:
- **Striker**: The batsman currently facing the ball
- **Non-striker**: The batsman at the other end
- **Rotate strike**: Striker and non-striker swap positions (happens on odd runs or end of over)

In [ ]:
def get_phase(over_number):
    """Determine the phase from the over number (1-indexed)."""
    if over_number <= 6:
        return "powerplay"
    elif over_number <= 15:
        return "middle"
    else:
        return "death"

# Pre-compute numpy arrays for extra runs sampling
_EXTRA_RUNS_ARR = np.array(EXTRA_RUNS_VALUES, dtype=np.int32)
_EXTRA_RUNS_P = np.array(EXTRA_RUNS_PROBS)
_RUN_VALUES_ARR = np.array(RUN_VALUES, dtype=np.int32)


def simulate_innings(batting_order, bowler_list, target=None, rng=None, detailed=False):
    """
    Simulate one T20 innings ball-by-ball.

    Parameters:
        batting_order (list): 11 batsman names in order
        bowler_list (list): 5 bowler names
        target (int or None): If chasing, stop when target is passed. None for 1st innings.
        rng: numpy random generator (for reproducibility)
        detailed (bool): If True, track ball_log and batsman_stats (slower).
                         If False, only track totals (fast mode for Monte Carlo).

    Returns:
        dict with total_runs, wickets, balls_faced, overs, run_rate,
        and optionally ball_log, batsman_stats, overs_runs
    """
    if rng is None:
        rng = np.random.default_rng()

    # --- State ---
    total_runs = 0
    wickets = 0
    legal_balls = 0
    next_batsman_idx = 2
    striker_idx = 0
    non_striker_idx = 1

    # Bowler rotation
    bowler_balls = [0] * len(bowler_list)
    current_bowler_idx = 0
    last_over_bowler_idx = -1

    # Tracking (only in detailed mode)
    overs_runs = []
    current_over_runs = 0

    if detailed:
        ball_log = []
        batsman_stats = {b: {"runs": 0, "balls": 0, "fours": 0, "sixes": 0, "out": False}
                         for b in batting_order}

    def pick_bowler():
        for i in range(len(bowler_list)):
            idx = (current_bowler_idx + i) % len(bowler_list)
            if idx != last_over_bowler_idx and bowler_balls[idx] < 24:
                return idx
        for idx in range(len(bowler_list)):
            if bowler_balls[idx] < 24:
                return idx
        return 0

    current_bowler_idx = pick_bowler()
    current_bowler = bowler_list[current_bowler_idx]

    # --- Ball-by-ball simulation ---
    while legal_balls < 120 and wickets < 10:
        over_number = (legal_balls // 6) + 1
        phase = get_phase(over_number)
        striker = batting_order[striker_idx]

        # ---- VARIABLE 3: Check for extras (Bernoulli) ----
        q = get_extras_q(current_bowler)
        if rng.random() < q:
            extra_runs = int(rng.choice(_EXTRA_RUNS_ARR, p=_EXTRA_RUNS_P))
            total_runs += extra_runs
            current_over_runs += extra_runs
            if detailed:
                ball_log.append({
                    "over": over_number, "ball": "extra", "phase": phase,
                    "batsman": striker, "bowler": current_bowler,
                    "runs": extra_runs, "event": "extra", "total": total_runs,
                    "wickets": wickets
                })
            continue

        # ---- VARIABLE 1: Check for wicket (Geometric / Bernoulli) ----
        p = get_survival_p(striker, phase)
        if rng.random() < p:
            wickets += 1
            legal_balls += 1
            bowler_balls[current_bowler_idx] += 1
            current_over_runs += 0

            if detailed:
                batsman_stats[striker]["balls"] += 1
                batsman_stats[striker]["out"] = True
                ball_log.append({
                    "over": over_number, "ball": (legal_balls - 1) % 6 + 1, "phase": phase,
                    "batsman": striker, "bowler": current_bowler,
                    "runs": 0, "event": "wicket", "total": total_runs,
                    "wickets": wickets
                })

            if next_batsman_idx < 11:
                striker_idx = next_batsman_idx
                next_batsman_idx += 1
            else:
                break

        else:
            # ---- VARIABLE 2: Sample runs (Categorical) ----
            scoring_probs = get_scoring_probs(striker, phase)
            sampled_runs = int(rng.choice(_RUN_VALUES_ARR, p=scoring_probs))

            # ---- VARIABLE 5: Strike rotation ----
            if sampled_runs <= 1:
                s = get_rotation_s(striker, phase)
                sampled_runs = 1 if rng.random() < s else 0

            total_runs += sampled_runs
            legal_balls += 1
            current_over_runs += sampled_runs
            bowler_balls[current_bowler_idx] += 1

            if detailed:
                batsman_stats[striker]["runs"] += sampled_runs
                batsman_stats[striker]["balls"] += 1
                if sampled_runs == 4:
                    batsman_stats[striker]["fours"] += 1
                elif sampled_runs == 6:
                    batsman_stats[striker]["sixes"] += 1
                event = {0: "dot", 1: "single", 2: "two", 3: "three", 4: "four", 6: "six"}.get(sampled_runs, "runs")
                ball_log.append({
                    "over": over_number, "ball": (legal_balls - 1) % 6 + 1, "phase": phase,
                    "batsman": striker, "bowler": current_bowler,
                    "runs": sampled_runs, "event": event, "total": total_runs,
                    "wickets": wickets
                })

            if sampled_runs % 2 == 1:
                striker_idx, non_striker_idx = non_striker_idx, striker_idx

        # ---- End of over? ----
        if legal_balls > 0 and legal_balls % 6 == 0:
            overs_runs.append(current_over_runs)
            current_over_runs = 0
            striker_idx, non_striker_idx = non_striker_idx, striker_idx
            last_over_bowler_idx = current_bowler_idx
            current_bowler_idx = pick_bowler()
            current_bowler = bowler_list[current_bowler_idx]

        # ---- Chase target reached? ----
        if target is not None and total_runs > target:
            break

    # Capture partial over
    if legal_balls % 6 != 0:
        overs_runs.append(current_over_runs)

    result = {
        "total_runs": total_runs,
        "wickets": wickets,
        "balls_faced": legal_balls,
        "overs": legal_balls / 6,
        "run_rate": total_runs / (legal_balls / 6) if legal_balls > 0 else 0,
        "overs_runs": overs_runs,
    }
    if detailed:
        result["ball_log"] = ball_log
        result["batsman_stats"] = batsman_stats
    return result


print("simulate_innings() defined — with fast mode for Monte Carlo.")

In [ ]:
def simulate_match(team_a_batting, team_a_bowlers, team_b_batting, team_b_bowlers, rng=None, detailed=False):
    """
    Simulate a full T20 match.

    Team A bats first (no target), Team B chases.
    Set detailed=True for ball_log and batsman_stats (slower).
    """
    if rng is None:
        rng = np.random.default_rng()

    # Team A bats first — bowled at by Team B's bowlers
    innings_1 = simulate_innings(team_a_batting, team_b_bowlers, target=None, rng=rng, detailed=detailed)

    # Team B chases — bowled at by Team A's bowlers
    innings_2 = simulate_innings(team_b_batting, team_a_bowlers, target=innings_1["total_runs"], rng=rng, detailed=detailed)

    # Determine winner
    if innings_2["total_runs"] > innings_1["total_runs"]:
        winner = "team_b"
    elif innings_2["total_runs"] < innings_1["total_runs"]:
        winner = "team_a"
    else:
        winner = "tie"

    return {
        "innings_1": innings_1,
        "innings_2": innings_2,
        "winner": winner,
    }


print("simulate_match() defined.")

In [ ]:
# ---- Quick test: simulate ONE match ----
# This verifies everything works before running 10,000 simulations

test = simulate_match(team_a_batting, team_a_bowlers, team_b_batting, team_b_bowlers, detailed=True)

inn1 = test["innings_1"]
inn2 = test["innings_2"]

print(f"=== Test Match ===")
print(f"{team_a_name}: {inn1['total_runs']}/{inn1['wickets']} in {inn1['overs']:.1f} overs (RR: {inn1['run_rate']:.2f})")
print(f"{team_b_name}: {inn2['total_runs']}/{inn2['wickets']} in {inn2['overs']:.1f} overs (RR: {inn2['run_rate']:.2f})")
print(f"Winner: {team_a_name if test['winner'] == 'team_a' else team_b_name if test['winner'] == 'team_b' else 'Tie'}")

# Show top scorers
print(f"\nTop scorers ({team_a_name}):")
for b, s in sorted(inn1["batsman_stats"].items(), key=lambda x: -x[1]["runs"])[:3]:
    if s["balls"] > 0:
        sr = s['runs'] / s['balls'] * 100
        print(f"  {b}: {s['runs']} off {s['balls']} balls (SR {sr:.1f})")

---
## Section 8: Monte Carlo Simulation (10,000 matches)

Now we run the simulation **10,000 times** to build up a picture of all possible outcomes.

### Why 10,000?
- Too few (100): results are noisy, win% could be off by 10%+
- 10,000: gives stable, reliable estimates (margin of error ~1%)
- More (100,000): diminishing returns, just takes longer

### Speed
We use **fast mode** (no ball-by-ball logging) — only totals are tracked. This makes each simulation ~50x faster than the detailed mode used for the test match above.

In [ ]:
# ============================================================
#  RUN 10,000 SIMULATIONS (fast mode — no ball logs)
# ============================================================

N_SIMULATIONS = 10_000
rng = np.random.default_rng(seed=42)  # Fixed seed for reproducibility

results = []
print(f"Simulating {N_SIMULATIONS:,} matches: {team_a_name} vs {team_b_name}...")
start = time.time()

for i in range(N_SIMULATIONS):
    result = simulate_match(team_a_batting, team_a_bowlers, team_b_batting, team_b_bowlers, rng=rng, detailed=False)
    results.append({
        "team_a_score": result["innings_1"]["total_runs"],
        "team_a_wickets": result["innings_1"]["wickets"],
        "team_a_rr": result["innings_1"]["run_rate"],
        "team_b_score": result["innings_2"]["total_runs"],
        "team_b_wickets": result["innings_2"]["wickets"],
        "team_b_rr": result["innings_2"]["run_rate"],
        "winner": result["winner"],
    })

    if (i + 1) % 2500 == 0:
        elapsed = time.time() - start
        rate = (i + 1) / elapsed
        print(f"  {i+1:,} / {N_SIMULATIONS:,} done... ({rate:.0f} matches/sec)")

elapsed = time.time() - start
df_results = pd.DataFrame(results)
print(f"\nDone! {N_SIMULATIONS:,} matches in {elapsed:.1f}s ({N_SIMULATIONS/elapsed:.0f} matches/sec)")

---
## Section 9: Simulation Results & Charts

In [ ]:
# ============================================================
#  RESULT 1: Win Probability
# ============================================================

win_counts = df_results["winner"].value_counts()
team_a_wins = win_counts.get("team_a", 0)
team_b_wins = win_counts.get("team_b", 0)
ties = win_counts.get("tie", 0)

print(f"=== Win Probability ({N_SIMULATIONS:,} simulations) ===")
print(f"  {team_a_name}: {team_a_wins/N_SIMULATIONS*100:.1f}% ({team_a_wins:,} wins)")
print(f"  {team_b_name}: {team_b_wins/N_SIMULATIONS*100:.1f}% ({team_b_wins:,} wins)")
print(f"  Ties:        {ties/N_SIMULATIONS*100:.1f}% ({ties:,})")

fig, ax = plt.subplots(figsize=(8, 4))
labels = [team_a_name, team_b_name, "Tie"]
values = [team_a_wins / N_SIMULATIONS * 100, team_b_wins / N_SIMULATIONS * 100, ties / N_SIMULATIONS * 100]
colors = ["#2196F3", "#F44336", "#9E9E9E"]
bars = ax.barh(labels, values, color=colors, edgecolor="black")
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f"{val:.1f}%",
            va='center', fontsize=12, fontweight='bold')
ax.set_xlabel("Win Probability (%)")
ax.set_title(f"Match Outcome: {team_a_name} vs {team_b_name}")
ax.set_xlim(0, max(values) + 10)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  RESULT 2: Score Distribution Histograms
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Team A scores
axes[0].hist(df_results["team_a_score"], bins=30, color="#2196F3", edgecolor="black", alpha=0.8)
axes[0].axvline(df_results["team_a_score"].mean(), color="red", linestyle="--",
                label=f"Mean: {df_results['team_a_score'].mean():.0f}")
axes[0].axvline(df_results["team_a_score"].median(), color="orange", linestyle="--",
                label=f"Median: {df_results['team_a_score'].median():.0f}")
axes[0].set_xlabel("Total Score")
axes[0].set_ylabel("Frequency")
axes[0].set_title(f"{team_a_name} — Score Distribution (Batting First)")
axes[0].legend()

# Team B scores
axes[1].hist(df_results["team_b_score"], bins=30, color="#F44336", edgecolor="black", alpha=0.8)
axes[1].axvline(df_results["team_b_score"].mean(), color="red", linestyle="--",
                label=f"Mean: {df_results['team_b_score'].mean():.0f}")
axes[1].axvline(df_results["team_b_score"].median(), color="orange", linestyle="--",
                label=f"Median: {df_results['team_b_score'].median():.0f}")
axes[1].set_xlabel("Total Score")
axes[1].set_ylabel("Frequency")
axes[1].set_title(f"{team_b_name} — Score Distribution (Chasing)")
axes[1].legend()

plt.tight_layout()
plt.show()

# Summary stats table
print(f"{'Metric':<20} {team_a_name:<20} {team_b_name:<20}")
print("-" * 60)
print(f"{'Mean Score':<20} {df_results['team_a_score'].mean():<20.1f} {df_results['team_b_score'].mean():<20.1f}")
print(f"{'Median Score':<20} {df_results['team_a_score'].median():<20.1f} {df_results['team_b_score'].median():<20.1f}")
print(f"{'Std Dev':<20} {df_results['team_a_score'].std():<20.1f} {df_results['team_b_score'].std():<20.1f}")
print(f"{'Min Score':<20} {df_results['team_a_score'].min():<20} {df_results['team_b_score'].min():<20}")
print(f"{'Max Score':<20} {df_results['team_a_score'].max():<20} {df_results['team_b_score'].max():<20}")
print(f"{'Avg Wickets':<20} {df_results['team_a_wickets'].mean():<20.1f} {df_results['team_b_wickets'].mean():<20.1f}")
print()
print("NOTE: The chasing team's mean score appears lower because successful chases")
print("stop early (e.g., chasing 170, they stop at 171 without batting full 20 overs).")
print("This is expected behaviour — compare win% for the true picture, not raw averages.")

In [ ]:
# ============================================================
#  RESULT 3: Simulated Run Rate Progression
# ============================================================
# Run one detailed match and plot the over-by-over run rate

detail = simulate_match(team_a_batting, team_a_bowlers, team_b_batting, team_b_bowlers,
                        rng=np.random.default_rng(seed=123), detailed=True)

fig, ax = plt.subplots(figsize=(12, 5))

overs_a = detail["innings_1"]["overs_runs"]
overs_b = detail["innings_2"]["overs_runs"]

x_a = range(1, len(overs_a) + 1)
x_b = range(1, len(overs_b) + 1)

ax.bar([x - 0.2 for x in x_a], overs_a, width=0.4, color="#2196F3", edgecolor="black",
       label=f"{team_a_name} ({detail['innings_1']['total_runs']}/{detail['innings_1']['wickets']})")
ax.bar([x + 0.2 for x in x_b], overs_b, width=0.4, color="#F44336", edgecolor="black",
       label=f"{team_b_name} ({detail['innings_2']['total_runs']}/{detail['innings_2']['wickets']})")

ax.axvspan(0.5, 6.5, alpha=0.05, color="blue")
ax.axvspan(6.5, 15.5, alpha=0.05, color="orange")
ax.axvspan(15.5, 20.5, alpha=0.05, color="red")

ax.set_xlabel("Over")
ax.set_ylabel("Runs in Over")
ax.set_title("Simulated Match — Over-by-Over Scoring")
ax.set_xticks(range(1, 21))
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 10: Validation Against IPL 2025

This is the most important section. We check: **does our simulation produce realistic results?**

### Approach:
1. Pick a real IPL 2025 match
2. Simulate that specific matchup 1000 times
3. Compare simulated outcomes against what actually happened

If our model is good, the actual score should fall within the simulated distribution.

In [ ]:
# Show available IPL 2025 matches for validation
val_matches = df_val.groupby(["match", "match_slug"]).agg(
    teams=('batting_team', lambda x: ' vs '.join(sorted(x.unique()))),
    total_balls=('runs_scored', 'count'),
).reset_index()
val_matches = val_matches[val_matches["total_balls"] > 100]  # Filter out abandoned

print("Available IPL 2025 matches for validation:")
print()
for i, (_, row) in enumerate(val_matches.iterrows(), 1):
    print(f"  {i:2d}. {row['match']:<20s} — {row['teams']}")

In [ ]:
# ---- Pick a match to validate ----
print("Enter the match number from the list above (default: 1):")
match_choice = input("Match number: ").strip()
match_idx = int(match_choice) - 1 if match_choice.isdigit() else 0
match_idx = max(0, min(match_idx, len(val_matches) - 1))

chosen_match = val_matches.iloc[match_idx]
chosen_slug = chosen_match["match_slug"]
chosen_title = chosen_match["match"]

print(f"\nSelected: {chosen_title} — {chosen_match['teams']}")

# Get actual match data
match_data = df_val[df_val["match_slug"] == chosen_slug].copy()

# Extract actual results
innings_1_data = match_data[match_data["innings"] == 1]
innings_2_data = match_data[match_data["innings"] == 2]

actual_team_a = innings_1_data["batting_team"].iloc[0]
actual_team_b = innings_2_data["batting_team"].iloc[0] if len(innings_2_data) > 0 else "Unknown"

actual_score_a = innings_1_data["total_runs"].iloc[-1] if len(innings_1_data) > 0 else 0
actual_wkts_a = innings_1_data["total_wickets"].iloc[-1] if len(innings_1_data) > 0 else 0
actual_score_b = innings_2_data["total_runs"].iloc[-1] if len(innings_2_data) > 0 else 0
actual_wkts_b = innings_2_data["total_wickets"].iloc[-1] if len(innings_2_data) > 0 else 0

print(f"\nActual Result:")
print(f"  {actual_team_a}: {actual_score_a}/{actual_wkts_a}")
print(f"  {actual_team_b}: {actual_score_b}/{actual_wkts_b}")
actual_winner = actual_team_b if int(actual_score_b) > int(actual_score_a) else actual_team_a
print(f"  Winner: {actual_winner}")

In [ ]:
# ---- Build teams from the validation match data ----
# Use the actual players who played in this match

val_team_a_batsmen = innings_1_data["batsman"].unique().tolist()
val_team_b_batsmen = innings_2_data["batsman"].unique().tolist()

# Get bowlers: who bowled in each innings
val_team_b_bowlers = innings_1_data["bowler"].unique().tolist()[:5]  # Bowled in inn 1 = team B bowlers
val_team_a_bowlers = innings_2_data["bowler"].unique().tolist()[:5]  # Bowled in inn 2 = team A bowlers

# Pad batting orders to 11
while len(val_team_a_batsmen) < 11:
    val_team_a_batsmen.append(val_team_a_batsmen[-1])
while len(val_team_b_batsmen) < 11:
    val_team_b_batsmen.append(val_team_b_batsmen[-1])
while len(val_team_a_bowlers) < 5:
    val_team_a_bowlers.append(val_team_a_bowlers[-1])
while len(val_team_b_bowlers) < 5:
    val_team_b_bowlers.append(val_team_b_bowlers[-1])

print(f"{actual_team_a} batting: {val_team_a_batsmen}")
print(f"{actual_team_a} bowling: {val_team_a_bowlers}")
print(f"{actual_team_b} batting: {val_team_b_batsmen}")
print(f"{actual_team_b} bowling: {val_team_b_bowlers}")

In [ ]:
# ---- Simulate this specific matchup 10,000 times ----

N_VAL = 10_000
rng_val = np.random.default_rng(seed=99)

val_results = []
print(f"Simulating {actual_team_a} vs {actual_team_b} x {N_VAL:,}...")
start = time.time()

for i in range(N_VAL):
    r = simulate_match(val_team_a_batsmen, val_team_a_bowlers,
                       val_team_b_batsmen, val_team_b_bowlers, rng=rng_val, detailed=False)
    val_results.append({
        "team_a_score": r["innings_1"]["total_runs"],
        "team_a_wickets": r["innings_1"]["wickets"],
        "team_b_score": r["innings_2"]["total_runs"],
        "team_b_wickets": r["innings_2"]["wickets"],
        "winner": r["winner"],
    })

    if (i + 1) % 2500 == 0:
        elapsed = time.time() - start
        rate = (i + 1) / elapsed
        print(f"  {i+1:,} done... ({rate:.0f} matches/sec)")

df_val_results = pd.DataFrame(val_results)
print(f"Done in {time.time()-start:.1f}s")

In [ ]:
# ============================================================
#  VALIDATION PLOT 1: Simulated Score Distribution vs Actual Score
# ============================================================
# The actual score should fall within the simulated distribution.
# The vertical red line = what actually happened.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Innings 1
axes[0].hist(df_val_results["team_a_score"], bins=30, color="#2196F3", edgecolor="black", alpha=0.7)
axes[0].axvline(int(actual_score_a), color="red", linewidth=2.5, linestyle="-",
                label=f"Actual: {actual_score_a}")
axes[0].axvline(df_val_results["team_a_score"].mean(), color="orange", linewidth=1.5, linestyle="--",
                label=f"Sim Mean: {df_val_results['team_a_score'].mean():.0f}")
axes[0].set_xlabel("Total Score")
axes[0].set_ylabel("Frequency")
axes[0].set_title(f"{actual_team_a} (Bat First) — Simulated vs Actual")
axes[0].legend(fontsize=11)

# Innings 2
axes[1].hist(df_val_results["team_b_score"], bins=30, color="#F44336", edgecolor="black", alpha=0.7)
axes[1].axvline(int(actual_score_b), color="red", linewidth=2.5, linestyle="-",
                label=f"Actual: {actual_score_b}")
axes[1].axvline(df_val_results["team_b_score"].mean(), color="orange", linewidth=1.5, linestyle="--",
                label=f"Sim Mean: {df_val_results['team_b_score'].mean():.0f}")
axes[1].set_xlabel("Total Score")
axes[1].set_ylabel("Frequency")
axes[1].set_title(f"{actual_team_b} (Chasing) — Simulated vs Actual")
axes[1].legend(fontsize=11)

plt.suptitle(f"Validation: {chosen_title} — {actual_team_a} vs {actual_team_b}", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Percentile of actual score in simulated distribution
pctile_a = (df_val_results["team_a_score"] <= int(actual_score_a)).mean() * 100
pctile_b = (df_val_results["team_b_score"] <= int(actual_score_b)).mean() * 100
print(f"{actual_team_a} actual score ({actual_score_a}) is at the {pctile_a:.0f}th percentile of simulated scores")
print(f"{actual_team_b} actual score ({actual_score_b}) is at the {pctile_b:.0f}th percentile of simulated scores")
print("(Good model: actual should fall between 10th-90th percentile)")

In [ ]:
# ============================================================
#  VALIDATION PLOT 2: Simulated Runs per Ball vs Actual
# ============================================================
# Compare the distribution of runs per delivery.

# Get actual runs distribution from the match
actual_runs_dist = match_data["runs_scored"].value_counts(normalize=True).sort_index()

# Get simulated runs distribution (from a single detailed sim)
detail_val = simulate_match(val_team_a_batsmen, val_team_a_bowlers,
                            val_team_b_batsmen, val_team_b_bowlers,
                            rng=np.random.default_rng(seed=42), detailed=True)
sim_runs = [b["runs"] for b in detail_val["innings_1"]["ball_log"] + detail_val["innings_2"]["ball_log"]]
sim_runs_dist = pd.Series(sim_runs).value_counts(normalize=True).sort_index()

# Align indices
all_runs = sorted(set(actual_runs_dist.index) | set(sim_runs_dist.index))
actual_vals = [actual_runs_dist.get(r, 0) * 100 for r in all_runs]
sim_vals = [sim_runs_dist.get(r, 0) * 100 for r in all_runs]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(all_runs))
ax.bar(x - 0.2, actual_vals, width=0.4, color="#F44336", edgecolor="black", label="Actual Match")
ax.bar(x + 0.2, sim_vals, width=0.4, color="#2196F3", edgecolor="black", label="Simulated")
ax.set_xlabel("Runs per Delivery")
ax.set_ylabel("Percentage (%)")
ax.set_title("Runs per Delivery — Actual vs Simulated")
ax.set_xticks(x)
ax.set_xticklabels([str(r) for r in all_runs])
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  VALIDATION PLOT 3: Wickets per Innings — Simulated vs Actual
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Innings 1 wickets
axes[0].hist(df_val_results["team_a_wickets"], bins=range(0, 12), color="#2196F3",
             edgecolor="black", alpha=0.7, align="left")
axes[0].axvline(int(actual_wkts_a), color="red", linewidth=2.5,
                label=f"Actual: {actual_wkts_a} wkts")
axes[0].set_xlabel("Wickets")
axes[0].set_ylabel("Frequency")
axes[0].set_title(f"{actual_team_a} — Wickets Distribution")
axes[0].legend()

# Innings 2 wickets
axes[1].hist(df_val_results["team_b_wickets"], bins=range(0, 12), color="#F44336",
             edgecolor="black", alpha=0.7, align="left")
axes[1].axvline(int(actual_wkts_b), color="red", linewidth=2.5,
                label=f"Actual: {actual_wkts_b} wkts")
axes[1].set_xlabel("Wickets")
axes[1].set_ylabel("Frequency")
axes[1].set_title(f"{actual_team_b} — Wickets Distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  VALIDATION PLOT 4: Run Rate by Over — Simulated vs Actual
# ============================================================

# Actual run rate per over
actual_rpo = innings_1_data.groupby("over_number")["runs_scored"].sum()

# Simulated run rate per over (from the detailed sim)
sim_overs = detail_val["innings_1"]["overs_runs"]

fig, ax = plt.subplots(figsize=(12, 5))

overs_x = range(1, min(len(actual_rpo), 20) + 1)
actual_y = [actual_rpo.get(o, 0) for o in overs_x]
sim_y = [sim_overs[o-1] if o-1 < len(sim_overs) else 0 for o in overs_x]

ax.plot(overs_x, actual_y, 'o-', color="#F44336", linewidth=2, markersize=8, label="Actual")
ax.plot(overs_x, sim_y, 's--', color="#2196F3", linewidth=2, markersize=8, label="Simulated")

ax.axvspan(0.5, 6.5, alpha=0.05, color="blue")
ax.axvspan(6.5, 15.5, alpha=0.05, color="orange")
ax.axvspan(15.5, 20.5, alpha=0.05, color="red")

ax.set_xlabel("Over")
ax.set_ylabel("Runs in Over")
ax.set_title(f"{actual_team_a} Innings — Run Rate per Over: Actual vs Simulated")
ax.set_xticks(range(1, 21))
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  VALIDATION PLOT 5: Predicted Win% vs Actual Outcome
# ============================================================

val_win_counts = df_val_results["winner"].value_counts()
pred_a_pct = val_win_counts.get("team_a", 0) / N_VAL * 100
pred_b_pct = val_win_counts.get("team_b", 0) / N_VAL * 100

fig, ax = plt.subplots(figsize=(8, 5))

labels = [actual_team_a, actual_team_b]
pred_pcts = [pred_a_pct, pred_b_pct]
actual_pcts = [100 if actual_winner == actual_team_a else 0,
               100 if actual_winner == actual_team_b else 0]

x = np.arange(len(labels))
ax.bar(x - 0.2, pred_pcts, width=0.4, color="#2196F3", edgecolor="black", label="Predicted Win%")
ax.bar(x + 0.2, actual_pcts, width=0.4, color="#F44336", edgecolor="black", label="Actual Outcome")

for i in range(len(labels)):
    ax.text(i - 0.2, pred_pcts[i] + 1, f"{pred_pcts[i]:.1f}%", ha='center', fontweight='bold')

ax.set_ylabel("Win Probability (%)")
ax.set_title(f"Predicted vs Actual — {chosen_title}")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.set_ylim(0, 110)
plt.tight_layout()
plt.show()

correct = (pred_a_pct > 50 and actual_winner == actual_team_a) or \
          (pred_b_pct > 50 and actual_winner == actual_team_b)
print(f"Prediction: {actual_team_a} {pred_a_pct:.1f}% vs {actual_team_b} {pred_b_pct:.1f}%")
print(f"Actual winner: {actual_winner}")
print(f"Prediction correct: {'YES' if correct else 'NO'}")

In [ ]:
# ============================================================
#  VALIDATION PLOT 6: Calibration Plot (across multiple matches)
# ============================================================
# Run quick simulations for several IPL 2025 matches and check
# if predicted probabilities are well-calibrated.

print("Running quick validation across multiple IPL 2025 matches...")
print("(200 sims per match for speed)\n")

calibration_data = []
rng_cal = np.random.default_rng(seed=77)

for _, mrow in val_matches.head(20).iterrows():
    m_data = df_val[df_val["match_slug"] == mrow["match_slug"]]
    inn1 = m_data[m_data["innings"] == 1]
    inn2 = m_data[m_data["innings"] == 2]

    if len(inn1) == 0 or len(inn2) == 0:
        continue

    t_a = inn1["batting_team"].iloc[0]
    t_b = inn2["batting_team"].iloc[0]
    act_s_a = int(inn1["total_runs"].iloc[-1])
    act_s_b = int(inn2["total_runs"].iloc[-1])
    act_win = "team_b" if act_s_b > act_s_a else "team_a"

    # Build lineups from match data
    ba = inn1["batsman"].unique().tolist()
    bb = inn2["batsman"].unique().tolist()
    bla = inn2["bowler"].unique().tolist()[:5]
    blb = inn1["bowler"].unique().tolist()[:5]
    while len(ba) < 11: ba.append(ba[-1])
    while len(bb) < 11: bb.append(bb[-1])
    while len(bla) < 5: bla.append(bla[-1])
    while len(blb) < 5: blb.append(blb[-1])

    wins_a = 0
    for _ in range(200):
        r = simulate_match(ba, bla, bb, blb, rng=rng_cal, detailed=False)
        if r["winner"] == "team_a":
            wins_a += 1

    pred_a = wins_a / 200
    calibration_data.append({
        "match": mrow["match"],
        "team_a": t_a, "team_b": t_b,
        "pred_a_win": pred_a,
        "actual_a_win": 1 if act_win == "team_a" else 0,
    })
    print(f"  {mrow['match']}: {t_a} pred {pred_a*100:.0f}% | actual {'WIN' if act_win=='team_a' else 'LOSS'}")

df_cal = pd.DataFrame(calibration_data)

# Calibration plot: bin predictions, compare to actual win rate
bins = [0, 0.3, 0.4, 0.5, 0.6, 0.7, 1.0]
df_cal["bin"] = pd.cut(df_cal["pred_a_win"], bins=bins)
cal_grouped = df_cal.groupby("bin", observed=True).agg(
    mean_pred=('pred_a_win', 'mean'),
    mean_actual=('actual_a_win', 'mean'),
    count=('actual_a_win', 'count'),
).dropna()

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], 'k--', label="Perfect calibration", linewidth=1.5)
ax.scatter(cal_grouped["mean_pred"], cal_grouped["mean_actual"],
           s=cal_grouped["count"] * 50, color="#2196F3", edgecolor="black", zorder=5)

for _, row in cal_grouped.iterrows():
    ax.annotate(f"n={int(row['count'])}", (row['mean_pred'] + 0.02, row['mean_actual'] - 0.03), fontsize=10)

ax.set_xlabel("Predicted Win Probability")
ax.set_ylabel("Actual Win Rate")
ax.set_title("Calibration Plot — Model Reliability")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

accuracy = (df_cal["pred_a_win"].round() == df_cal["actual_a_win"]).mean() * 100
print(f"\nOverall accuracy (picking the favourite): {accuracy:.0f}% across {len(df_cal)} matches")

In [ ]:
# ============================================================
#  VALIDATION PLOTS 7 & 8: Batsman-level accuracy
# ============================================================
# Compare simulated vs actual batting average and strike rate
# for the top batsmen in our validation match.

# Get actual per-batsman stats from the match
actual_bat_stats = match_data.groupby("batsman").agg(
    actual_runs=('runs_scored', 'sum'),
    actual_balls=('runs_scored', 'count'),
).reset_index()
actual_bat_stats = actual_bat_stats[actual_bat_stats["actual_balls"] >= 5]  # Min 5 balls
actual_bat_stats["actual_sr"] = (actual_bat_stats["actual_runs"] / actual_bat_stats["actual_balls"] * 100).round(1)

# Run 100 detailed sims and average per-batsman stats
sim_bat_totals = {}
rng_bat = np.random.default_rng(seed=55)
for _ in range(100):
    r = simulate_match(val_team_a_batsmen, val_team_a_bowlers,
                       val_team_b_batsmen, val_team_b_bowlers, rng=rng_bat, detailed=True)
    for inn_key in ["innings_1", "innings_2"]:
        for batter, stats in r[inn_key]["batsman_stats"].items():
            if batter not in sim_bat_totals:
                sim_bat_totals[batter] = {"runs": 0, "balls": 0, "innings": 0}
            sim_bat_totals[batter]["runs"] += stats["runs"]
            sim_bat_totals[batter]["balls"] += stats["balls"]
            if stats["balls"] > 0:
                sim_bat_totals[batter]["innings"] += 1

sim_bat_df = pd.DataFrame([
    {"batsman": b, "sim_avg_runs": d["runs"]/max(d["innings"],1),
     "sim_sr": d["runs"]/max(d["balls"],1)*100}
    for b, d in sim_bat_totals.items() if d["balls"] > 50
])

# Merge actual and simulated
compare = actual_bat_stats.merge(sim_bat_df, on="batsman", how="inner")

if len(compare) >= 3:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Plot 7: Runs comparison
    axes[0].scatter(compare["actual_runs"], compare["sim_avg_runs"], s=100, color="#2196F3", edgecolor="black")
    max_val = max(compare["actual_runs"].max(), compare["sim_avg_runs"].max()) + 10
    axes[0].plot([0, max_val], [0, max_val], 'k--', alpha=0.5, label="Perfect match")
    for _, row in compare.iterrows():
        axes[0].annotate(row["batsman"].split()[-1], (row["actual_runs"]+1, row["sim_avg_runs"]+1), fontsize=9)
    axes[0].set_xlabel("Actual Runs (this match)")
    axes[0].set_ylabel("Simulated Average Runs")
    axes[0].set_title("Batsman Runs — Actual vs Simulated")
    axes[0].legend()

    # Plot 8: Strike rate comparison
    axes[1].scatter(compare["actual_sr"], compare["sim_sr"], s=100, color="#F44336", edgecolor="black")
    max_sr = max(compare["actual_sr"].max(), compare["sim_sr"].max()) + 20
    axes[1].plot([0, max_sr], [0, max_sr], 'k--', alpha=0.5, label="Perfect match")
    for _, row in compare.iterrows():
        axes[1].annotate(row["batsman"].split()[-1], (row["actual_sr"]+2, row["sim_sr"]+2), fontsize=9)
    axes[1].set_xlabel("Actual Strike Rate (this match)")
    axes[1].set_ylabel("Simulated Strike Rate")
    axes[1].set_title("Batsman Strike Rate — Actual vs Simulated")
    axes[1].legend()

    plt.suptitle(f"Player-Level Validation — {chosen_title}", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough overlapping batsmen for scatter plot. Try a match with more common players.")

---
## Section 11: Summary & Interpretation

### What we built
A **Monte Carlo simulation engine** that models IPL T20 cricket matches ball-by-ball using 5 random variables:

| Variable | Distribution | What it models |
|----------|-------------|----------------|
| X — Batsman survival | Geometric(p) | When a batsman gets out |
| R — Runs per ball | Categorical | Scoring pattern per batsman |
| W — Extras | Bernoulli(q) | Wides/no-balls per bowler |
| E — Extra runs | Categorical | Runs from extras |
| S — Strike rotation | Bernoulli(s) | Singles vs dots tendency |

### Key assumptions
- Each ball is **independent** (no momentum/pressure effects)
- Player ability is **stationary** (same across the season)
- Phase (PP/Middle/Death) captures tactical changes
- Historical IPL 2024 stats represent current ability

### Limitations
- Pitch/weather conditions not modelled
- Toss advantage not included
- No fatigue/momentum effects
- Players with little historical data use league averages

### What could be improved
- Add pitch type as a feature
- Model batsman-bowler specific matchups
- Include "new batsman vulnerability" (higher dismissal rate in first 6 balls)
- Use multiple seasons for more stable parameter estimates

---
## Credits

**Data Source:**
- [ESPNcricinfo](https://www.espncricinfo.com/) — Ball-by-ball cricket data

**Data Library:**
- [`cricdata`](https://github.com/arnavbonigala/cricdata) by Arnav Bonigala — Python wrapper for ESPNcricinfo data. MIT License.

**AI Assistance:**
- This notebook was developed with assistance from [Claude](https://claude.ai) by Anthropic.

**References:**
- Kimber, A.C. and Hansford, A.R. (1993). "A Statistical Analysis of Batting in Cricket." *Journal of the Royal Statistical Society.* — The classic paper showing batting scores follow an exponential distribution.

---
*Built for the Simulation of Cricket Match assignment.*